# Scenario 2 — A Decentrally-Controlled Door (workflows + live state)

This notebook demonstrates a door PLC wrapped by the KAPPS Semantic Middleware, exposing:
- Two **workflows** (`open`, `close`) — invokable via capability-mediated operations, like scenario 1.
- One **StateProperty** (`status`) — served live over HTTP from in-memory state.

**Key insight:** high-frequency state (door position, temperature, robot pose) must NOT flood the knowledge graph. Instead the graph holds only a stable *endpoint pointer*; the actual value lives in the process and is fetched on demand.

Following **ADR 0010**, this notebook is self-contained: it connects to GraphDB from environment variables, clears a dedicated test repository, loads only the needed ontology (`svc:` + demo), seeds the door resource, then runs the complete scenario. It never touches production state.

In [1]:
import os
import sys
import threading
import time
from pathlib import Path

import httpx
import uvicorn
from rdflib.namespace import RDF

from graph_db_interface import GraphDB
from kapps_ogm import OGM
from kapps_semantic_middleware import SemanticMiddleware
from kapps_semantic_middleware.registration import (
    mint_capability_iri,
    mint_state_property_iri,
    mint_workflow_iri,
)
from kapps_semantic_middleware.vocabulary import SVC

# seed.py sits beside this notebook in examples/. Make it importable regardless of cwd.
for _cand in (Path.cwd(), Path.cwd() / "examples"):
    if (_cand / "seed.py").exists():
        sys.path.insert(0, str(_cand))
        break
import seed  # noqa: E402

db = GraphDB.from_env()
print(f"Connected to GraphDB: {os.getenv('GRAPHDB_URL')} / {os.getenv('GRAPHDB_REPOSITORY')}")

INFO:KafkaManager:KafkaManager initialized


INFO:GraphDB:Using GraphDB repository 'Tests' as user 'etienneh'.


Connected to GraphDB: https://graphdb.iam-mms.kit.edu / Tests


## Step 1 — Seed a clean repository (door ontology + resource)

`seed_scenario2()` clears the repository, loads the `svc:` module plus the demo ontology (which contains the door classes), and creates the door resource individual — the ground-truth schema before any middleware registration.

In [2]:
seed.seed_scenario2(db)
print("Repository cleared and seeded with the door ontology.")
print(f"  Door resource:      {seed.DOOR_RESOURCE}")
print(f"  Door service class: {seed.DOOR_SERVICE_CLASS}")

Repository cleared and seeded with the door ontology.
  Door resource:      https://example.org/kapps-demo#door_042
  Door service class: https://example.org/kapps-demo#DoorControllerService


## Step 2 — The door's in-memory state and its handlers

The door keeps its status in a plain Python dict (`handlers._door`). The workflows mutate it; the state getter reads it. **This value never touches the graph** — it lives only in the middleware process and is served live over HTTP on request. The handlers live in `handlers.py` (not a notebook cell) because `@workflow` type-checks each function by reading its module source, which a Jupyter cell does not have.

In [3]:
import handlers  # noqa: E402
from handlers import door_close, door_open, door_status  # noqa: E402

print(f"Door handlers imported. Initial state: {handlers._door['status']!r}")

Door handlers imported. Initial state: 'closed'


## Step 3 — Start the door middleware with two `@workflow` and one `@state`

We construct the middleware for the door resource, register the open/close workflows and the status state property, then start uvicorn in a background thread. Registration writes the workflow and state-property *structure* to the graph (IRIs, types, endpoints) — but never the actual state value.

In [4]:
DOOR_PORT = 8997

service_iri = seed.DOOR_RESOURCE + "_service"
open_wf = mint_workflow_iri(service_iri, "door_open")
close_wf = mint_workflow_iri(service_iri, "door_close")
status_sp = mint_state_property_iri(service_iri, "door_status")

mw = SemanticMiddleware(
    mode="resource",
    resource_iri=seed.DOOR_RESOURCE,
    service_class=seed.DOOR_SERVICE_CLASS,
    ogm=OGM(db=db),
    host="127.0.0.1",
    port=DOOR_PORT,
)

mw.workflow(
    capability_class=seed.DOOR_OPEN_CAPABILITY_CLASS,
    workflow_class=seed.DOOR_OPEN_WORKFLOW_CLASS,
)(door_open)
mw.workflow(
    capability_class=seed.DOOR_CLOSE_CAPABILITY_CLASS,
    workflow_class=seed.DOOR_CLOSE_WORKFLOW_CLASS,
)(door_close)
mw.state(
    capability_class=seed.DOOR_STATUS_CAPABILITY_CLASS,
    state_property_class=seed.DOOR_STATUS_STATE_CLASS,
)(door_status)


def _start_server(mw, port):
    config = uvicorn.Config(mw.app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    t0 = time.time()
    while not server.started and time.time() - t0 < 30:
        time.sleep(0.05)
    if not server.started:
        raise RuntimeError("server did not start")
    return server, thread


server, thread = _start_server(mw, DOOR_PORT)
base = f"http://127.0.0.1:{DOOR_PORT}"

print(f"Middleware started on port {DOOR_PORT}")
print(f"  open workflow:  {open_wf}")
print(f"  close workflow: {close_wf}")
print(f"  status state:   {status_sp}")

Middleware started on port 8997
  open workflow:  https://example.org/kapps-demo#door_042_service_workflow_door_open
  close workflow: https://example.org/kapps-demo#door_042_service_workflow_door_close
  status state:   https://example.org/kapps-demo#door_042_service_state_door_status


### 🔎 The middleware's live REST API (Swagger UI)

The door middleware is now a running FastAPI server. Open its interactive Swagger UI at
**http://127.0.0.1:8997/docs** to see the two workflow endpoints
(`POST /workflows/door_open/execute`, `POST /workflows/door_close/execute`) and the state
endpoint (`GET /state/door_status`), and try them directly. The server stays up until the
shutdown cell (Step 8). (On a remote host, forward port 8997 as well as the Jupyter port.)

In [5]:
print("Swagger UI:", f"{base}/docs")
print("Routes:", sorted(r.path for r in mw.app.routes if "workflows" in r.path or "state" in r.path))

Swagger UI: http://127.0.0.1:8997/docs
Routes: ['/state/door_status', '/workflows/door_close/description', '/workflows/door_close/execute', '/workflows/door_close/execute_background', '/workflows/door_close/interrupt', '/workflows/door_open/description', '/workflows/door_open/execute', '/workflows/door_open/execute_background', '/workflows/door_open/interrupt']


## Step 4 — What registration wrote (workflows + a state ENDPOINT, not a value)

Links are materialized on the instance-owned (inverse) side (ADR 0008): a Workflow knows its Service via `svc:isWorkflowOf`, a StateProperty via `svc:isStatePropertyOf`; the container-side `svc:hasWorkflow`/`svc:hasStateProperty` are OWL-inferable and not written. The state property carries its type and an `svc:endpoint` pointer. **There is no "opened"/"closed" literal anywhere** — only structural metadata and the endpoint.

In [6]:
assert db.triple_exists((open_wf, SVC.isWorkflowOf, service_iri))
assert db.triple_exists((close_wf, SVC.isWorkflowOf, service_iri))
print("Workflows registered:")
print(f"  {open_wf} --isWorkflowOf--> {service_iri}")
print(f"  {close_wf} --isWorkflowOf--> {service_iri}")

assert db.triple_exists((status_sp, SVC.isStatePropertyOf, service_iri))
assert db.triple_exists((status_sp, RDF.type, seed.DOOR_STATUS_STATE_CLASS))
print("\nState property registered:")
print(f"  {status_sp} --isStatePropertyOf--> {service_iri}")
for _, _, obj in db.triples_get(sub=status_sp, pred=SVC.endpoint):
    print(f"  {status_sp} --svc:endpoint--> {obj}")

status_cap = mint_capability_iri(seed.DOOR_RESOURCE, "door_status")
assert db.triple_exists((status_cap, SVC.providedByStateProperty, status_sp))
print(f"  {status_cap} --providedByStateProperty--> {status_sp}")
print("\nAll structural triples present; no state VALUE in the graph.")

Workflows registered:
  https://example.org/kapps-demo#door_042_service_workflow_door_open --isWorkflowOf--> https://example.org/kapps-demo#door_042_service
  https://example.org/kapps-demo#door_042_service_workflow_door_close --isWorkflowOf--> https://example.org/kapps-demo#door_042_service

State property registered:
  https://example.org/kapps-demo#door_042_service_state_door_status --isStatePropertyOf--> https://example.org/kapps-demo#door_042_service
  https://example.org/kapps-demo#door_042_service_state_door_status --svc:endpoint--> http://127.0.0.1:8997/state/door_status
  https://example.org/kapps-demo#door_042_capability_door_status --providedByStateProperty--> https://example.org/kapps-demo#door_042_service_state_door_status

All structural triples present; no state VALUE in the graph.


## Step 5 — Read the live status over HTTP (served from memory)

The `GET /state/door_status` endpoint returns the current value from the in-memory `_door` dict by invoking the registered `door_status()` handler — it never queries the graph. Initially the door is closed.

In [7]:
r = httpx.get(f"{base}/state/door_status")
print(f"GET {base}/state/door_status  ->  {r.status_code}  {r.json()!r}")
assert r.status_code == 200 and r.json() == "closed"

INFO:httpx:HTTP Request: GET http://127.0.0.1:8997/state/door_status "HTTP/1.1 200 OK"


GET http://127.0.0.1:8997/state/door_status  ->  200  'closed'


## Step 6 — Open the door via its Operation, then re-read the live status

We create an Operation implementing the `door_open` capability and execute it through the middleware (top-level `await`, Jupyter). The workflow mutates `_door["status"]` in memory; a subsequent GET reflects the change immediately — still without touching the graph. Then we close it the same way.

In [8]:
seed.create_operation(
    db, seed.DOOR_OPEN_OPERATION, mint_capability_iri(seed.DOOR_RESOURCE, "door_open")
)
res_open = await mw.execute(seed.DOOR_OPEN_OPERATION)  # noqa: F704
print(f"execute(open):  success={res_open['success']} result={res_open['result']!r}")
assert res_open["success"] and res_open["result"] == "opened"
print(f"GET status  ->  {httpx.get(f'{base}/state/door_status').json()!r}")
assert httpx.get(f"{base}/state/door_status").json() == "opened"

seed.create_operation(
    db, seed.DOOR_CLOSE_OPERATION, mint_capability_iri(seed.DOOR_RESOURCE, "door_close")
)
res_close = await mw.execute(seed.DOOR_CLOSE_OPERATION)  # noqa: F704
print(f"execute(close): success={res_close['success']} result={res_close['result']!r}")
assert res_close["success"] and res_close["result"] == "closed"
print(f"GET status  ->  {httpx.get(f'{base}/state/door_status').json()!r}")
assert httpx.get(f"{base}/state/door_status").json() == "closed"

INFO:httpx:HTTP Request: POST http://127.0.0.1:8997/workflows/door_open/execute "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET http://127.0.0.1:8997/state/door_status "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET http://127.0.0.1:8997/state/door_status "HTTP/1.1 200 OK"


execute(open):  success=True result='opened'
GET status  ->  'opened'


INFO:httpx:HTTP Request: POST http://127.0.0.1:8997/workflows/door_close/execute "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET http://127.0.0.1:8997/state/door_status "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET http://127.0.0.1:8997/state/door_status "HTTP/1.1 200 OK"


execute(close): success=True result='closed'
GET status  ->  'closed'


## Step 7 — Confirm the live value was never persisted

We iterate every triple on the state-property IRI and verify none of the objects is the literal "opened"/"closed". The graph contains only structural metadata (type, links, endpoint); the high-frequency value stays exclusively in memory.

In [9]:
sp_predicates = {str(p) for _, p, _ in db.triples_get(sub=status_sp)}
print("Predicates on the state property:")
for p in sorted(sp_predicates):
    print(f"  {p}")
assert str(SVC.endpoint) in sp_predicates

for _, _, obj in db.triples_get(sub=status_sp):
    assert str(obj) not in ("opened", "closed"), "state value must not be persisted"
print("\nConfirmed: no 'opened'/'closed' literal in the graph — served live from memory only.")

Predicates on the state property:
  http://www.w3.org/1999/02/22-rdf-syntax-ns#type
  https://w3id.org/circularfactory/Service#endpoint
  https://w3id.org/circularfactory/Service#isStatePropertyOf
  https://w3id.org/circularfactory/Service#providesCapability

Confirmed: no 'opened'/'closed' literal in the graph — served live from memory only.


## Step 8 — Shutdown

We stop the server. Deregistration removes the state property's endpoint (the live service is gone) but preserves the individual and its type for audit/history.

In [10]:
server.should_exit = True
thread.join(timeout=20)
time.sleep(0.5)

endpoint_after = list(db.triples_get(sub=status_sp, pred=SVC.endpoint))
print(f"State endpoint removed on shutdown: {len(endpoint_after) == 0}")
assert db.triple_exists((status_sp, RDF.type, seed.DOOR_STATUS_STATE_CLASS))
print(f"State property individual preserved: True")

State endpoint removed on shutdown: True
State property individual preserved: True


## Recap

This scenario demonstrates the **multi-database** principle at the heart of KAPPS:

| Concern | Storage | Rationale |
|---|---|---|
| Discovery & coordination (workflows, capabilities, state endpoints) | Knowledge graph | Stable, queryable, shared substrate |
| High-frequency state (door position, sensor readings) | In-memory / external store | Avoid flooding the graph; serve live on demand |

The graph is the *coordination* layer, not the *data* layer. By registering only an endpoint pointer for a state property, KAPPS enables real-time state access without compromising graph performance or consistency — a pattern that scales to many rapidly-changing devices, all discoverable via the graph, all serving live values from their native stores.